# Label Creation
Label y = 1 if |z-score| decreases over the next 5 trading days (spread moves toward zero).

Labels use future data by construction — this is intentional for supervised learning.
The key discipline is that labels are only used *for training* on in-sample data.

In [1]:
import sys, numpy as np, pandas as pd
sys.path.append('..')

In [2]:
IS_END    = '2023-12-31'
OOS_START = '2024-01-01'
N_AHEAD   = 5   # prediction horizon: 5 trading days

In [3]:
selected_pairs = pd.read_csv('../data_processed/selected_pairs.csv', index_col=0)

for _, row in selected_pairs.iterrows():
    t1, t2   = row['ticker1'], row['ticker2']
    features = pd.read_csv(f'../data_processed/features_{t1}_{t2}.csv',
                           index_col=0, parse_dates=True)

    z = features['z_score']
    # y=1 when |z| shrinks over the next N days => spread is mean-reverting
    future_abs_z = z.abs().shift(-N_AHEAD)
    y = (future_abs_z < z.abs()).astype(int).rename('y')

    dataset = features.join(y).dropna()
    dataset.to_csv(f'../data_processed/dataset_{t1}_{t2}.csv')

    is_n  = (dataset.index <= IS_END).sum()
    oos_n = (dataset.index >= OOS_START).sum()
    print(f'  {t1}/{t2}: total={len(dataset)} | IS={is_n} | OOS={oos_n} | y=1 rate={y.mean():.3f}')

print('Label creation complete.')

  EWJ/HYG: total=1256 | IS=754 | OOS=502 | y=1 rate=0.521
  HYG/XLC: total=1256 | IS=754 | OOS=502 | y=1 rate=0.504
  XLU/XOP: total=1256 | IS=754 | OOS=502 | y=1 rate=0.518
  HYG/MTUM: total=1256 | IS=754 | OOS=502 | y=1 rate=0.492
  EWC/IWD: total=1256 | IS=754 | OOS=502 | y=1 rate=0.513
  GDXJ/LQD: total=1256 | IS=754 | OOS=502 | y=1 rate=0.492
  USO/XLU: total=1256 | IS=754 | OOS=502 | y=1 rate=0.509
  EWG/HYG: total=1256 | IS=754 | OOS=502 | y=1 rate=0.508
  EWJ/XLC: total=1256 | IS=754 | OOS=502 | y=1 rate=0.513
  EWC/XLF: total=1256 | IS=754 | OOS=502 | y=1 rate=0.490
  GDXJ/USO: total=1256 | IS=754 | OOS=502 | y=1 rate=0.490
  XLP/XLV: total=1256 | IS=754 | OOS=502 | y=1 rate=0.501
  XLP/XLU: total=1256 | IS=754 | OOS=502 | y=1 rate=0.522


  EWA/XLB: total=1256 | IS=754 | OOS=502 | y=1 rate=0.518
  GDX/USO: total=1256 | IS=754 | OOS=502 | y=1 rate=0.497
Label creation complete.
